# Sprint 5 — Main Experiment & Results Analysis
## Від перевіреної реалізації до експериментальних результатів і обережної оцінки H1

**AI Research Practices · Науково-дослідна практика · Спринт 5 із 7**

**Вхід:** `Research_Proposal_v1.md` (Sprint 2), `Experimental_Protocol_v1.md` (Sprint 3), зафіксовані моделі, код та `Implementation_Report_v1.md` з успішним пілотом (Sprint 4).

**Мета:** виконати **основні вимірювання за погодженим протоколом**, перевірити повноту та цілісність результатів, оцінити варіативність шумових реалізацій, зіставити спостереження з RQ1–RQ3 і обережно оцінити підтримку H1.

**Результат:** `Experimental_Results_v1.md`, таблиці вимірювань, візуалізації, конфігурація запуску та відтворювані артефакти у персональному GitHub fork.

**Межа Sprint 5:** моделі **не перенавчаємо**, гіперпараметри, метрики та рівні шуму **не підбираємо за результатами тестування**. Ми ще не пишемо повну наукову статтю: її підготовка належить наступним етапам.

**Наскрізний приклад:** три зафіксовані у Sprint 4 моделі Logistic Regression (Baseline Model), Random Forest, Gradient Boosting; шум тільки у стандартизованих **тестових ознаках**; фіксований Training/Test split; macro-F1; `σ ∈ {0, 0.1, 0.2, 0.4, 0.6}` та seed `11, 23, 37, 41, 53` для додатних `σ`.

**Перед запуском:** необхідно успішно виконати Sprint 4 і мати доступ до його каталогу `research_artifacts` із `manifest.json`, `split_indices.npz`, `model_*.joblib` та `pilot_results.csv`. Не завантажуємо неперевірені файли моделей з чужих джерел.

## Маршрут спринту

1. Відновлюємо точні умови Experimental Protocol і перевіряємо готовність артефактів Sprint 4.
2. Розгортаємо матрицю 15 умов у **63 заплановані оцінки метрики** без дублювання пілоту.
3. Виконуємо вимірювання за спільних реалізацій шуму для всіх алгоритмів.
4. Перевіряємо повноту, правильність та відтворюваність результатів.
5. Аналізуємо `Q(m,σ)`, `D(m,σ)` і варіативність за seed; будуємо графіки.
6. Зіставляємо фактичні спостереження з RQ1–RQ3, визначаємо межі оцінки H1.
7. Документуємо Experimental Results та готуємо матеріали до наступного спринту.

**Ключова навичка:** відрізняти *отриманий результат*, *його невизначеність* і *допустимий науковий висновок*.

## A. Problem Situation — чи достатньо запустити весь цикл і порівняти цифри?

У Sprint 4 ми довели, що код працює: за `σ=0.2` і `seed=11` отримали по одному пілотному `Q` та `D` для трьох моделей. Але пілот **не відповідає** на питання про профілі деградації та їхню мінливість.

Тепер маємо перейти від одного випробування до **заздалегідь спланованої серії**. При цьому необхідно з'ясувати:

- Чи справді всі 15 умов виконані та чи є по п'ять реалізацій для кожного додатного `σ`?
- Чи не змінилися навчена модель, Test Set або спосіб стандартизації?
- Як порівнювати *абсолютну якість* із *величиною її зменшення*?
- Що означає розкид результатів і чого він **не** дозволяє стверджувати?
- Як повідомити про підтримку H1, не видаючи обмежений експеримент за універсальний висновок?

**Проблема спринту:** коректно виконаний код ще не гарантує коректної інтерпретації; між даними й науковим висновком потрібний контроль якості результатів.

## B1. Essential Theory — від Experimental Matrix до даних вимірювання

У Sprint 3 визначили **експериментальну умову** як поєднання алгоритму `m` та інтенсивності шуму `σ`. Для нашої H1 це **3 × 5 = 15 умов**. Однак за кожного додатного `σ` маємо **п'ять випадкових реалізацій шуму**; отже, таблиця фактичних вимірювань міститиме:

- `3` значення `Q(m,0)` — по одному для кожної фіксованої моделі;
- `3 × 4 × 5 = 60` оцінок за шуму;
- **разом 63 рядки оцінювання метрики**.

Одиницею запису є комбінація `(model, sigma, seed)` для `σ>0` або `(model, sigma=0)` для чистої оцінки. До кожного шумового рядка додаємо `Q_clean`, `Q_noisy` і `D=Q_clean−Q_noisy`.

**Важливо:** 63 рядки — **не 63 незалежні датасети**. Усі результати отримано на одному Test Set; при цьому варіативність за seed відображає лише випадковість доданого шуму. 

**Пілот Sprint 4** (`σ=0.2`, `seed=11`) належить запланованій матриці. Якщо пілот виконано без зміни протоколу та помилок, переносимо його **рівно один раз**. Повторний обчислювальний контроль тієї самої умови не створює нової незалежної реалізації.

## B2. Essential Theory — якість, деградація та спільні реалізації шуму

Для кожної моделі `m` зафіксовано її **Clean Reference Performance** `Q(m,0)` — macro-F1 на Test Set без штучного шуму. Для додатного `σ` визначаємо:

$$
D(m,\sigma)=Q(m,0)-Q(m,\sigma).
$$

- `Q(m,σ)` відповідає на питання про **абсолютну якість** за даної інтенсивності шуму.
- `D(m,σ)` показує **втрату якості відносно власного чистого рівня**. Якщо `D<0`, спостережена якість підвищилася; це не доказ користі шуму.

Для кожної пари `(σ, seed)` створюємо **одну матрицю спотворень**, спільну для всіх трьох моделей. Це забезпечує однакові умови тестування: відмінності між результатами алгоритмів не пов’язані з використанням різних випадкових спотворень.

Повторення різних seed потрібні для оцінювання варіативності внаслідок шуму. Зміна seed не означає ні нового Test Set, ні нового навчання моделей.

**Особливість Pipeline зі Sprint 4:** шум вносимо **після StandardScaler** і передаємо стандартизовані ознаки безпосередньо навченому класифікатору. Повторно запускати `StandardScaler` для вже стандартизованих шумових ознак не можна.

## B3. Essential Theory — як узагальнюємо повторні вимірювання

Для кожної пари `(m,σ>0)` маємо п'ять значень `Q` та п'ять відповідних значень `D`. Показуємо **середнє арифметичне** та **вибіркове стандартне відхилення** (`ddof=1`).

$$
\overline{Q}(m,\sigma)=\frac{1}{R}\sum_{r=1}^{R}Q(m,\sigma,r),\qquad R=5.
$$

За `σ=0` маємо **одну** чисту оцінку, а не п'ять штучно повторених однакових спостережень. Стандартне відхилення для цього випадку позначаємо як **не застосовується**.

**Межа інтерпретації:** вибірковий розкид п'яти реалізацій шуму на **одному** Test Set не є довірчим інтервалом узагальнення на інші датасети. Не трактуємо ці 63 вимірювання як незалежні вибірки; висновки про статистичну значущість потребують окремої заздалегідь погодженої методики, що враховує залежності між вимірюваннями.

## B4. Essential Theory — перевірка результатів перед інтерпретацією

До аналізу H1 перевіряємо **якість даних експерименту**:

1. **Повнота:** присутні всі умови та запланована кількість реалізацій.
2. **Унікальність:** немає дубльованих ключів `(model, sigma, seed)`.
3. **Цілісність:** усі `Q` належать інтервалу `[0,1]`, усі `D` відповідають різниці `Q_clean−Q_noisy`.
4. **Спільність умов:** та сама матриця шуму застосована до всіх алгоритмів за однакових `(σ,seed)`.
5. **Відтворюваність:** версії моделей і конфігурація не змінилися, пілот узгоджений із записом Sprint 4.
6. **Прозорість:** зберігаємо всі результати, а не лише найбільш переконливі значення чи графіки.

Якщо перевірка не пройшла, спочатку встановлюємо причину. **Не змінюємо постфактум протокол, щоб отримати очікувану форму кривих**. Виправлення технічних помилок і повторення пов'язаних вимірювань документуємо.

## B5. Essential Theory — від спостереження до RQ і H1

- **RQ1:** як змінюється `Q(m,σ)` для кожного алгоритму? Показуємо криві середніх значень і розкид.
- **RQ2:** як відрізняються профілі деградації `D(m,σ)` між алгоритмами? Порівнюємо залежності зменшення якості від інтенсивності шуму та визначаємо характер спостережуваних відмінностей.
- **RQ3:** наскільки чутливий результат до конкретної випадкової реалізації шуму? Аналізуємо розкид і незвичні випадки, не приховуючи їх.

Для H1 розрізняємо **підтримку**, **відсутність достатньої підтримки** й **неоднозначність**. Ці оцінки залежать від критеріїв, визначених *до* результатів. Якщо в протоколі не задано конкретний поріг практично значущої відмінності чи інферентну процедуру, робимо **описовий висновок** і прямо вказуємо, чого він не доводить.

**Межі висновків** не змінилися зі Sprint 3: конкретний датасет, три оптимізовані за чистим macro-F1 моделі, фіксований split, тільки адитивний гаусівський шум у стандартизованих тестових ознаках і визначені рівні `σ`. Результати не поширюємо автоматично на інші типи шуму, датасети чи умови навчання.

## C1. Guided Research Example — відновлюємо експеримент Sprint 4

**Умова:** запускаємо комірки цього прикладу після завершення Sprint 4, у робочому середовищі з його каталогом `research_artifacts`. Звідти завантажуємо конфігурацію, індекси розбиття й три **вже навчені** моделі. `GridSearchCV` повторно не викликаємо.

Шлях до артефактів можна змінити відповідно до середовища. Завантажувати `joblib`-моделі можна лише з **власних довірених артефактів**, оскільки десеріалізація сторонніх файлів небезпечна.

In [ ]:
from pathlib import Path
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from sklearn.datasets import load_wine
from sklearn.metrics import f1_score

# Підключаємо лише перевірені артефакти Sprint 4.
ARTIFACT_DIR = Path("research_artifacts")
RESULT_DIR = ARTIFACT_DIR / "sprint5"
required = ["manifest.json", "split_indices.npz", "pilot_results.csv",
            "model_LR.joblib", "model_RF.joblib", "model_GB.joblib"]
missing = [name for name in required if not (ARTIFACT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Артефакти Sprint 4 не знайдено: {missing}. "
        "Виконаємо Sprint 4 або уточнимо ARTIFACT_DIR."
    )
manifest = json.loads((ARTIFACT_DIR / "manifest.json").read_text(encoding="utf-8"))
CONFIG = manifest["config"]
assert CONFIG["dataset"] == "sklearn.datasets.load_wine"
assert CONFIG["noise_levels"] == [0.0, 0.1, 0.2, 0.4, 0.6]
assert CONFIG["noise_seeds"] == [11, 23, 37, 41, 53]
assert CONFIG["pilot_sigma"] == 0.2 and CONFIG["pilot_seed"] == 11

# Результати Sprint 5 не перезаписують файли моделей і пілоту Sprint 4.
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Знайдено погоджену конфігурацію Sprint 4.")

## C2. Guided Research Example — перевіряємо split, моделі та Clean Reference

Відновлюємо той самий набір даних і **збережені індекси**, а не створюємо новий split. Звіряємо чисту якість із `manifest.json`; якщо вона не збігається, основний експеримент не запускаємо. Так само перевіряємо спільний scaler і той самий маршрут прогнозування, що працював у Sprint 4.

In [ ]:
# Відновлюємо Test Set з індексів, зафіксованих у Sprint 4.
raw = load_wine(as_frame=True)
X, y = raw.data.copy(), raw.target.copy()
with np.load(ARTIFACT_DIR / "split_indices.npz") as split:
    train_idx, test_idx = split["train"], split["test"]
assert len(X) == manifest["dataset_rows"]
assert X.columns.tolist() == manifest["feature_names"]
assert set(train_idx).isdisjoint(set(test_idx))
X_test, y_test = X.iloc[test_idx].copy(), y.iloc[test_idx].copy()

# Завантажуємо три зафіксовані моделі; без повторного навчання.
models = {name: joblib.load(ARTIFACT_DIR / f"model_{name}.joblib")
          for name in ("LR", "RF", "GB")}
clean_reference = manifest["clean_reference_macro_f1"]
reference_scaler = models["LR"].named_steps["scale"]
for name, pipeline in models.items():
    scaler = pipeline.named_steps["scale"]
    assert np.allclose(scaler.mean_, reference_scaler.mean_)
    assert np.allclose(scaler.scale_, reference_scaler.scale_)
    score = f1_score(y_test, pipeline.predict(X_test), average="macro")
    assert np.isclose(score, clean_reference[name], atol=1e-12), name
z_test_clean = reference_scaler.transform(X_test)

# Прогнозуємо вже стандартизовані ознаки, минаючи повторний scaler.
def predict_standardized(pipeline, z):
    return pipeline.named_steps["model"].predict(z)
for name, pipeline in models.items():
    assert np.array_equal(pipeline.predict(X_test),
                          predict_standardized(pipeline, z_test_clean))
print("Моделі, split і Q(m,0) узгоджені зі Sprint 4:", clean_reference)

## C3. Guided Research Example — відтворюємо пілот без дублювання

Пілот `σ=0.2`, `seed=11` уже виконано в Sprint 4. Перед його включенням до основної таблиці перевіряємо, що новий розрахунок за тими самими моделями й шумом **дає ті самі показники**. До матриці записуємо лише один результат на модель.

Функція генерації шуму повторює визначення зі Sprint 4: новий генератор `np.random.default_rng(seed)` і незалежне нормальне спотворення кожного елемента стандартизованої тестової матриці.

In [ ]:
# Визначаємо спільний генератор шуму відповідно до Sprint 4.
def add_gaussian_noise(z, sigma, seed):
    a = np.asarray(z, dtype=float)
    if a.ndim != 2 or not np.isfinite(a).all() or sigma < 0 or not np.isfinite(sigma):
        raise ValueError("Недопустима матриця ознак або sigma")
    rng = np.random.default_rng(int(seed))
    return a + rng.normal(loc=0.0, scale=float(sigma), size=a.shape)

# Повторно ОБЧИСЛЮЄМО пілот лише для перевірки, а не як нову умову матриці.
pilot_saved = pd.read_csv(ARTIFACT_DIR / "pilot_results.csv", index_col="model")
assert set(pilot_saved.index) == set(models)
z_pilot = add_gaussian_noise(z_test_clean, CONFIG["pilot_sigma"], CONFIG["pilot_seed"])
for name, pipeline in models.items():
    q_pilot = f1_score(y_test, predict_standardized(pipeline, z_pilot), average="macro")
    assert np.isclose(q_pilot, pilot_saved.loc[name, "Q_noisy"], atol=1e-10)
    assert np.isclose(clean_reference[name] - q_pilot,
                      pilot_saved.loc[name, "D"], atol=1e-10)
print("Пілотні оцінки відтворено; в основній таблиці не дублюємо.")

## C4. Guided Research Example — виконуємо повну Experimental Matrix

Для кожного додатного `σ` генеруємо п'ять реалізацій шуму. **Для кожної пари `(σ,seed)` створюємо спільну матрицю тестових ознак, і всі три алгоритми отримують саме її.** Не навчаємо моделі заново. `σ=0` записуємо один раз на модель.

Пілотну умову включаємо зі збережених результатів Sprint 4; решту вимірювань обчислюємо. У підсумку повинно бути **63 унікальні рядки**.

In [ ]:
# 3 чисті оцінки + 60 шумових оцінок, одна з яких для кожної моделі вже була пілотом.
rows = []
for name in models:
    rows.append({"model": name, "sigma": 0.0, "seed": pd.NA,
                 "Q_clean": clean_reference[name],
                 "Q_noisy": clean_reference[name], "D": 0.0,
                 "source": "clean_reference_sprint4"})

for sigma in CONFIG["noise_levels"][1:]:
    for seed in CONFIG["noise_seeds"]:
        is_pilot = (sigma == CONFIG["pilot_sigma"] and seed == CONFIG["pilot_seed"])
        # Одна й та сама матриця шуму для всіх трьох моделей.
        z_noisy = None if is_pilot else add_gaussian_noise(z_test_clean, sigma, seed)
        for name, pipeline in models.items():
            if is_pilot:
                q = float(pilot_saved.loc[name, "Q_noisy"])
                origin = "pilot_sprint4"
            else:
                prediction = predict_standardized(pipeline, z_noisy)
                q = float(f1_score(y_test, prediction, average="macro"))
                origin = "main_sprint5"
            rows.append({"model": name, "sigma": float(sigma), "seed": int(seed),
                         "Q_clean": float(clean_reference[name]),
                         "Q_noisy": q,
                         "D": float(clean_reference[name]) - q,
                         "source": origin})

measurements = pd.DataFrame(rows)
measurements["seed"] = measurements["seed"].astype("Int64")
measurements = measurements.sort_values(["model", "sigma", "seed"],
                                        na_position="first").reset_index(drop=True)
print("Кількість оцінок:", len(measurements))
print(measurements.head(10).to_string(index=False))

## C5. Guided Research Example — перевіряємо повноту й коректність вимірювань

До побудови графіків перевіримо, що маємо саме заплановану матрицю. Окремо переконаємося, що чистий результат не множився на кількість seed і що для пілотної умови використано по одному запису на модель.

In [ ]:
# Перевіряємо 15 умов, 63 оцінки, 5 реалізацій для кожного додатного sigma.
assert len(measurements) == 63
assert measurements.groupby(["model", "sigma"]).ngroups == 15
assert not measurements.duplicated(["model", "sigma", "seed"]).any()
for name in models:
    subset = measurements.loc[measurements.model == name]
    assert len(subset.loc[subset.sigma == 0]) == 1
    for sigma in CONFIG["noise_levels"][1:]:
        condition = subset.loc[subset.sigma == sigma]
        assert sorted(condition.seed.tolist()) == sorted(CONFIG["noise_seeds"])
assert measurements.Q_noisy.between(0, 1).all()
assert measurements.Q_clean.between(0, 1).all()
assert np.allclose(measurements.D, measurements.Q_clean - measurements.Q_noisy)
assert int((measurements.source == "pilot_sprint4").sum()) == 3
assert int((measurements.source == "clean_reference_sprint4").sum()) == 3
print("PASS: 15 умов, 63 унікальні оцінки, 3 пілотні записи, коректні Q і D.")

## C6. Guided Research Example — узагальнюємо результати за рівнями шуму

Для кожного `(model,σ)` обчислюємо середній `Q`, середній `D` та вибіркові стандартні відхилення за **п'ятьма** шумовими реалізаціями. Для чистого рівня `σ=0` стандартне відхилення залишаємо невизначеним, оскільки маємо одну оцінку.

**Це описова статистика для фіксованого Test Set**, а не інтервали узагальнення на довільні нові датасети.

In [ ]:
# Узагальнюємо повтори без штучного дублювання чистої оцінки.
summary = (measurements.groupby(["model", "sigma"], as_index=False)
           .agg(n=("Q_noisy", "size"),
                Q_mean=("Q_noisy", "mean"), Q_sd=("Q_noisy", "std"),
                D_mean=("D", "mean"), D_sd=("D", "std")))
assert len(summary) == 15
assert summary.loc[summary.sigma == 0, "Q_sd"].isna().all()
assert (summary.loc[summary.sigma > 0, "n"] == 5).all()
print(summary.round(4).to_string(index=False))

## C7. Guided Research Example — візуалізуємо RQ1 і RQ2

На першому графіку показуємо середній `Q(m,σ)` за рівнями шуму, на другому — середній `D(m,σ)`. Вертикальні відрізки — **±1 вибіркове стандартне відхилення між шумовими реалізаціями**, а **не довірчі інтервали**. За `σ=0` відрізків розкиду немає: одну чисту оцінку не можна видавати за п'ять повторень.

Під час інтерпретації відокремлюємо високу абсолютну якість від невеликої втрати якості. **Графіки окремих реалізацій шуму** побудуємо в C8.

In [ ]:
# Будуємо окремі графіки для абсолютної якості та деградації.
for metric, spread, ylabel, filename in [
    ("Q_mean", "Q_sd", "Macro-F1, Q", "rq1_quality.png"),
    ("D_mean", "D_sd", "Зменшення якості, D", "rq2_degradation.png"),
]:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for name in ("LR", "RF", "GB"):
        data = summary[summary.model == name].sort_values("sigma")
        # Нульова похибка лише для графічного відображення чистої оцінки.
        yerr = data[spread].fillna(0).to_numpy()
        ax.errorbar(data.sigma, data[metric], yerr=yerr,
                    marker="o", capsize=3, label=name)
    ax.set(xlabel="Інтенсивність тестового шуму σ", ylabel=ylabel)
    ax.grid(alpha=0.25)
    ax.legend(title="Алгоритм")
    fig.tight_layout()
    fig.savefig(RESULT_DIR / filename, dpi=160)
    plt.show()

## C8. Guided Research Example — графіки окремих реалізацій деградації (RQ3)

У C7 ми побачили **середні** залежності `Q(m,σ)` та `D(m,σ)` для трьох моделей. Проте за кожного додатного рівня шуму маємо **п'ять фактичних оцінок**, отриманих із різними seed. Середня крива не показує, наскільки ці результати відрізняються між собою.

Тому для кожного алгоритму побудуємо окремий графік: **тонкі лінії** показують `D(m,σ)` для п'яти реалізацій шуму, **товста лінія** — середнє `D_mean(m,σ)`. За `σ=0` усі криві починаються з `D=0`, бо штучного шуму ще немає.

Ці графіки допомагають відповісти на **RQ3**: оцінити, наскільки залежність деградації змінюється від конкретної випадкової реалізації. Це **описова варіативність на одному Test Set**, а не доказ узагальнюваності на інші набори даних.


In [ ]:
# Показуємо фактичні траєкторії деградації для кожного seed і кожної моделі.
variability = summary.loc[summary.sigma > 0,
                          ["model", "sigma", "n", "Q_mean", "Q_sd", "D_mean", "D_sd"]]
assert len(variability) == 12
assert (variability.n == 5).all()
print("Варіативність Q та D за п'ятьма реалізаціями шуму:")
print(variability.round(4).to_string(index=False))

# Окремий графік для кожного алгоритму: 5 seed та середня деградація.
for name in ("LR", "RF", "GB"):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    subset = measurements[(measurements.model == name) & (measurements.sigma > 0)]
    for seed in CONFIG["noise_seeds"]:
        series = subset[subset.seed == seed].sort_values("sigma")
        xs = np.r_[0.0, series.sigma.to_numpy()]
        ds = np.r_[0.0, series.D.to_numpy()]
        ax.plot(xs, ds, marker="o", linewidth=1.1, alpha=0.55,
                label=f"seed={seed}")
    mean_series = summary[summary.model == name].sort_values("sigma")
    ax.plot(mean_series.sigma, mean_series.D_mean,
            color="black", linewidth=2.8, marker="o", label="Середня D")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set(title=f"{name}: реалізації деградації за різних рівнів шуму",
           xlabel="Інтенсивність тестового шуму σ", ylabel="Деградація D")
    ax.grid(alpha=0.25)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / f"rq3_variability_{name}.png", dpi=160)
    plt.show()


## C9. Guided Research Example — інтерпретуємо фактичні результати для RQ та H1

Після C6–C8 у нас є **реальні результати виконаного експерименту**: таблиця `summary.csv` (формується в C10), графіки середніх `Q` та `D` із C7 і графіки окремих реалізацій деградації з C8. Не замінюємо ці дані умовними числами.

Нижче послідовно **покажемо приклад на фактично обчислених значеннях `summary`**. Код формує стислий аналітичний виклад із чисел поточного запуску; отже, при повторному виконанні текст залишиться узгодженим із результатами.

### 1. RQ1 — як змінюється якість класифікації?

Спочатку розглядаємо графік `Q(m,σ)` із C7. Для кожного алгоритму зіставляємо **власний** `Q(m,0)` із середньою якістю за `σ=0.6` і перевіряємо, що відбувається на **проміжних рівнях**. Можливе окреме підвищення якості через шум не слід приховувати.

### 2. RQ2 — як відрізняються профілі деградації?

Далі розглядаємо графік `D(m,σ)` із C7. Порівнюємо не тільки кінцеві значення, а й форму кривих: за яких рівнів шуму втрати зростають швидше, чи мають алгоритми різну початкову якість та чи однаково реагують на сильніший шум. **Менша деградація не тотожна вищій абсолютній якості.**

### 3. RQ3 — наскільки результати залежать від реалізації шуму?

Графіки C8 показують для кожного алгоритму п'ять окремих кривих деградації та середню криву. Порівнюємо їхній розкид за різних `σ` із табличними `D_sd` і `Q_sd`. **Це варіативність від випадкового шуму на тому самому Test Set**, а не оцінка мінливості між незалежними датасетами.

### 4. Який висновок допустимий щодо H1?

Описові відмінності між середніми кривими можуть бути **сумісними з H1** про різні профілі деградації. Проте без наперед визначених критеріїв і коректної статистичної процедури ми **не оголошуємо статистичну значущість**, не доводимо універсальної переваги алгоритму й не поширюємо висновок на інші дані чи типи шуму.

Наступна комірка виводить **приклад готового описового аналізу на наших фактичних результатах**. Він відповідає саме умовам цього навчального запуску, а не підміняє наукового висновку для власної теми студента.


In [ ]:
# Пояснюємо RQ1–RQ3 на ФАКТИЧНИХ результатах цього запуску.
from IPython.display import Markdown, display

names = {"LR": "Logistic Regression", "RF": "Random Forest", "GB": "Gradient Boosting"}
max_sigma = max(CONFIG["noise_levels"])

# RQ1: початкова та кінцева середня якість, усі проміжні рівні перевірені.
lines = ["### Результати нашого експерименту", "**RQ1 — якість:**"]
for name in ("LR", "RF", "GB"):
    s = summary[summary.model == name].sort_values("sigma")
    q_start = float(s.iloc[0].Q_mean)
    q_end = float(s.iloc[-1].Q_mean)
    differences = np.diff(s.Q_mean.to_numpy())
    profile = "невисхідна на всіх досліджених рівнях" if np.all(differences <= 1e-12) else "немонотонна на окремих рівнях"
    lines.append(f"- **{names[name]}:** Q(0) = {q_start:.4f}; "
                 f"середня Q({max_sigma:g}) = {q_end:.4f}; крива {profile}.")

# RQ2: зіставлення всіх профілів, зокрема деградації на кінцевому рівні.
lines += ["", "**RQ2 — профілі деградації:**"]
end = summary[np.isclose(summary.sigma, max_sigma)].set_index("model")
for name in ("LR", "RF", "GB"):
    lines.append(f"- **{names[name]}:** середня D({max_sigma:g}) = "
                 f"{float(end.loc[name, 'D_mean']):.4f}.")
dvalues = end.D_mean.to_numpy()
lines.append("- За цим рівнем шуму найбільшу середню деградацію має "
             f"**{names[end.D_mean.idxmax()]}**, найменшу — **{names[end.D_mean.idxmin()]}**. "
             "Для відповіді про *профілі* враховуємо також усі проміжні точки графіка C7.")

# RQ3: спостережений розкид на фіксованому Test Set.
lines += ["", "**RQ3 — варіативність:**"]
for name in ("LR", "RF", "GB"):
    sd = float(end.loc[name, "D_sd"])
    lines.append(f"- **{names[name]}:** вибіркове SD деградації за "
                 f"σ={max_sigma:g} становить {sd:.4f}; окремі реалізації показано на графіку C8.")
display(Markdown("\n".join(lines)))


**Щодо H1**: Для цих даних і за визначеного механізму тестового шуму спостерігаємо описові відмінності між профілями деградації трьох моделей. Це узгоджується з припущенням H1 на рівні спостережень, але не є формальним статистичним підтвердженням: у цьому прикладі не встановлено окремого наперед погодженого критерію істотності відмінностей. Результати обмежені обраним датасетом, фіксованим split, моделями, рівнями й механізмом шуму.

## C10. Guided Research Example — зберігаємо результат і його походження

Результати зберігаємо як `measurements.csv`, `summary.csv`, два графіки середніх залежностей, три графіки варіативності та `results_manifest.json`. У маніфесті фіксуємо **SHA-256 файлів моделей**, щоб надалі виявити їхню непомітну заміну. У цьому спринті **не перезаписуємо** `manifest.json` і моделі Sprint 4.

CSV містить числові результати; наукову інтерпретацію вписуємо до `Experimental_Results_v1.md` за окремим шаблоном. Не додаємо вигаданих пояснень до фактичних вимірювань.

In [ ]:
# Зберігаємо усі вимірювання, таблиці, маніфест і контрольні хеші моделей.
measurements.to_csv(RESULT_DIR / "measurements.csv", index=False, float_format="%.12g")
summary.to_csv(RESULT_DIR / "summary.csv", index=False, float_format="%.12g")

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

results_manifest = {
    "sprint": 5,
    "protocol": "Experimental_Protocol_v1.md (Sprint 3)",
    "sprint4_manifest": "../manifest.json",
    "model_sha256": {name: sha256_file(ARTIFACT_DIR / f"model_{name}.joblib")
                     for name in models},
    "config": CONFIG,
    "model_count": 3,
    "unique_conditions": 15,
    "measurement_rows": len(measurements),
    "pilot_condition_reused_once_per_model": True,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__,
    "files": ["measurements.csv", "summary.csv",
              "rq1_quality.png", "rq2_degradation.png",
              "rq3_variability_LR.png", "rq3_variability_RF.png",
              "rq3_variability_GB.png"],
    "interpretation_scope": "descriptive_fixed_test_set_only",
}
(RESULT_DIR / "results_manifest.json").write_text(
    json.dumps(results_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Збережено:", sorted(p.name for p in RESULT_DIR.iterdir()))

## Тренувальна вправа 1 — 15 умов чи 63 незалежні спостереження?

**Ситуація:** для трьох моделей визначили п'ять рівнів `σ`, з яких чотири додатні. Для кожної додатної інтенсивності виконуємо п'ять випадкових реалізацій шуму на одному Test Set.

1. Обчислюємо кількість унікальних умов і кількість рядків оцінювання метрики.
2. Пояснюємо, чому ці рядки не є незалежними датасетами.
3. З'ясовуємо, що станеться з кількістю рядків, якщо помилково записати пілот двічі.

Відповідь для самоперевірки наведено після контрольних питань у розділі G.

## Тренувальна вправа 2 — якість і деградація

**Ситуація (умовні числа, не результати нашого датасету):** для A: `Q(A,0)=0.92`, `Q(A,0.2)=0.82`; для B: `Q(B,0)=0.84`, `Q(B,0.2)=0.79`.

1. Обчислюємо `D(A,0.2)` і `D(B,0.2)`.
2. Визначаємо, яка модель має вищу абсолютну якість і яка — меншу деградацію.
3. Пояснюємо, чому за одним вимірюванням не можна стверджувати про відмінність *профілів*.

Відповідь для самоперевірки наведено після контрольних питань у розділі G.

## Тренувальна вправа 3 — інтерпретація варіативності

**Ситуація:** за `σ=0.4` для моделі A отримано п’ять значень macro-F1: `0.80`, `0.82`, `0.78`, `0.81`, `0.79`. Усі оцінки виконано на одному Test Set з різними seed.

1. Обчислюємо середнє значення macro-F1.
2. Пояснюємо, що саме відображає розкид цих п’яти значень.
3. Визначаємо, чи достатньо таких повторень, щоб узагальнити результат на інші датасети, і обґрунтовуємо відповідь.

Відповідь для самоперевірки наведено після контрольних питань у розділі G.

## Тренувальна вправа 4 — від результатів до Research Questions

**Ситуація (умовні числа).** Дві моделі оцінено на тому самому Test Set. Для моделі A маємо `Q(A,0)=0.90`, `Q_mean(A,0.4)=0.78`, `Q_sd(A,0.4)=0.02`. Для моделі B: `Q(B,0)=0.88`, `Q_mean(B,0.4)=0.80`, `Q_sd(B,0.4)=0.03`.

1. Обчислюємо `D_mean(A,0.4)` та `D_mean(B,0.4)`.
2. Формулюємо **одне речення** про порівняння якості та деградації за `σ=0.4`.
3. Пояснюємо, що показує `Q_sd` і чому за одним рівнем шуму ще не можна описати повний профіль деградації.

Відповідь для самоперевірки наведено після контрольних питань у розділі G.


## D. Independent Research — основний експеримент за власним протоколом

Повертаємося до **власного** `Experimental_Protocol_v1.md` і працездатної реалізації зі Sprint 4. Набір алгоритмів, характер впливу, метрики та кількість повторень із навчального прикладу **не є обов'язковими для інших тем**.

1. Перед запуском перевіряємо статус `READY FOR EXPERIMENT`, усуваємо невирішені технічні помилки.
2. Завантажуємо зафіксовані моделі / процедури та конфігурацію без повторного tuning.
3. Виконуємо **всі** умови й повторення власної Experimental Matrix, зберігаючи також незручні або несподівані значення.
4. Відстежуємо походження даних, seed, версії середовища й результати кожного вимірювання.
5. Перевіряємо повноту таблиці та правильність обчислення показників.
6. Аналізуємо метрики, розкид і порівняння відповідно до Research Questions.
7. Формулюємо обережну оцінку гіпотези в межах наперед визначених критеріїв і Scope of Conclusions.
8. Документуємо відхилення від Protocol, рішення щодо виправлень і зв'язок із результатами.
9. Формуємо `Experimental_Results_v1.md`, графіки й відтворювані артефакти у **власному fork**.

**Підсумковий продукт:** Experimental Results v1.0. Не підмінюємо його ні навчальним прикладом, ні готовою статтею.

### D1. Protocol → Results Map

| Research Question | Умова / ID протоколу | Дані вимірювання | Аналіз / графік | Критерій інтерпретації |
|---|---|---|---|---|
| RQ1 | [ ] | [ ] | [ ] | [ ] |
| RQ2 | [ ] | [ ] | [ ] | [ ] |
| RQ3 (за потреби) | [ ] | [ ] | [ ] | [ ] |

**Пояснення:** [чи є всі вимірювання, заплановані в протоколі, і як вони допоможуть оцінити RQ?]

### D2. Реєстр експериментальних вимірювань

**Версії датасету, моделей, конфігурації та коду:** [заповнюємо]

**Експериментальні умови / повторення / спосіб реєстрації:** [заповнюємо]

**Як обробляємо пілот Sprint 4 без подвійного підрахунку:** [заповнюємо]

**Докази незмінності зафіксованих моделей і метрик:** [заповнюємо]

**Розташування повної таблиці вимірювань:** [заповнюємо]

**Пояснення:** [які вимірювання не вдалися та як це задокументовано?]

### D3. Аналіз і візуалізація

| RQ | Основне спостереження з посиланням на дані | Варіативність / невизначеність | Межа висновку |
|---|---|---|---|
| RQ1 | [ ] | [ ] | [ ] |
| RQ2 | [ ] | [ ] | [ ] |
| RQ3 (за потреби) | [ ] | [ ] | [ ] |

**Зміст графіків і таблиць:** [заповнюємо]

**Як забезпечено порівнюваність умов:** [заповнюємо]

**Чому вибраний аналіз відповідає Experimental Protocol:** [заповнюємо]

### D4. Валідність, H1 та межі висновків

**Загрози валідності, які залишилися після перевірок:** [заповнюємо]

**Критерії оцінювання H1, зафіксовані в протоколі:** [заповнюємо]

**Спостереження, що підтримують / не підтримують / не дозволяють оцінити H1:** [заповнюємо]

**Допустима область висновків (Scope of Conclusions):** [заповнюємо]

**Що потребуватиме окремого експерименту в майбутньому:** [заповнюємо]

### D5. Формуємо Experimental Results v1.0

Результати D1–D4 переносимо до `Experimental_Results_v1.md` за файлом `Experimental_Results_v1_Template.md`. Зберігаємо **повну таблицю вимірювань, агреговані таблиці, графіки, маніфест і відомості про середовище**.

Перевіряємо послідовність:

**Experimental Protocol → умова → вимірювання → перевірка → аналіз → RQ → оцінка H1 → Scope of Conclusions.**

Якщо виявлено відхилення, документуємо його причину та наслідки. Не приховуємо невдалих запусків і не змінюємо критерії лише заради привабливішого результату.

## E1. Sprint Review — перевірка навчального прикладу

- [ ] Завантажено саме моделі, split та конфігурацію зі Sprint 4; tuning не повторювали.
- [ ] Чиста якість `Q(m,0)` відтворюється на тому самому Test Set.
- [ ] Шум внесено **після стандартизації**, без повторного `StandardScaler`.
- [ ] За однакового `σ` і seed для всіх моделей використано спільну реалізацію шуму.
- [ ] Пілот `σ=0.2`, `seed=11` повторно перевірено і внесено до матриці один раз.
- [ ] Отримано 15 умов і **63 унікальні оцінки**.
- [ ] Перевірено коректність `Q`, `D`, ключів і кількості повторень.
- [ ] Показано графіки середніх `Q`, `D`, окремі криві деградації за п’ятьма seed та інтерпретацію фактичних результатів для RQ1–RQ3.
- [ ] Відмінність між описовим аналізом і статистичним узагальненням зрозуміла.

**Результат:** повний навчальний експеримент виконано; таблиці та графіки збережено.

## E2. Sprint Review — перевірка власного дослідження

- [ ] Основні вимірювання відповідають власному Experimental Protocol v1.0.
- [ ] Завершено всі заздалегідь визначені умови та повторення або документовано причини відхилень.
- [ ] Програмний код, моделі, Test Set і метрики не підлаштовано під результати.
- [ ] Пілотні вимірювання враховано без дублювання, якщо це дозволяє протокол.
- [ ] Таблиця результатів повна, перевірена й придатна до відтворення.
- [ ] Здійснено аналіз для кожного Research Question.
- [ ] Оцінено варіативність і обмеження обраної методики аналізу.
- [ ] Висновок щодо H1 має належну силу і не виходить за межі наявних доказів.
- [ ] `Experimental_Results_v1.md`, таблиці, графіки, конфігурація і README збережені у персональному fork.
- [ ] Суперечності, обмеження та питання для наукового керівника задокументовано.

**Статус:** `READY FOR RESEARCH REPORT`, якщо матеріали достатні для написання звіту / рукопису; інакше `REVISION NEEDED`. Самоперевірка не замінює наукового перегляду керівником.

## E3. Контрольні питання

1. Чим основний експеримент Sprint 5 відрізняється від пілоту Sprint 4?
2. Чому у матриці 15 умов, але 63 оцінки?
3. Чому пілот не слід рахувати двічі?
4. Чому Test Set і моделі не змінюємо після tuning?
5. Навіщо застосовуємо однакову реалізацію шуму до всіх алгоритмів?
6. Що характеризують `Q(m,σ)` і `D(m,σ)`? Чому порівнюємо обидва?
7. Яку варіативність відображає стандартне відхилення за п'ятьма seed?
8. Чому п'ять seed не означають п'ять незалежних датасетів?
9. Що відрізняє описові похибки на графіку від довірчих інтервалів?
10. Які перевірки виконуємо перед інтерпретацією результатів?
11. Як зіставляємо отримані вимірювання з RQ1, RQ2 та RQ3?
12. За яких умов можна обґрунтовано сказати, що результати підтримують H1?
13. Чому відсутність підтримки H1 не доводить еквівалентності алгоритмів?
14. Які файли й параметри потрібні для відтворення Experiment Results?

## G. Solutions to Training Exercises — відповіді для самоперевірки

### Вправа 1

Унікальних умов: `3 × 5 = 15`. Рядків: `3 + 3 × 4 × 5 = 63`. Це не незалежні вибірки, оскільки Test Set та моделі залишаються фіксованими. Якщо три пілотні записи для `σ=0.2, seed=11` помилково повторити, отримаємо **66 рядків**, із яких три є дублікатами.

### Вправа 2

`D(A,0.2)=0.92−0.82=0.10`; `D(B,0.2)=0.84−0.79=0.05`. A має вищу абсолютну якість за шуму (`0.82 > 0.79`), B — меншу деградацію (`0.05 < 0.10`). Для висновку про *профілі* необхідно порівняти серію рівнів шуму й врахувати повторення.

### Вправа 3

Середнє macro-F1 дорівнює `(0.80 + 0.82 + 0.78 + 0.81 + 0.79) / 5 = 0.80`. Розкид відображає залежність оцінки якості від конкретної реалізації випадкового шуму за фіксованого рівня `σ=0.4`. Ці повторення виконані на одному Test Set, тому не доводять узагальнюваність результатів на інші незалежні датасети.

### Вправа 4

`D_mean(A,0.4)=0.90−0.78=0.12`; `D_mean(B,0.4)=0.88−0.80=0.08`. За `σ=0.4` модель B має вищу середню якість (`0.80 > 0.78`) і меншу деградацію (`0.08 < 0.12`), хоча без шуму її якість була нижчою. `Q_sd` відображає варіативність серед повторних реалізацій шуму на тому самому Test Set: у B вона більша (`0.03 > 0.02`). Для висновку про профіль потрібні вимірювання за кількох рівнів `σ`, а для узагальнень — відповідний протокол.


## Підсумок і перехід до Sprint 6

**Sprint 1:** визначаємо проблему, H1 та Research Questions.

**Sprint 2:** критично аналізуємо публікації, формуємо Research Gap та Proposal.

**Sprint 3:** погоджуємо Experimental Protocol, матрицю умов, показники й межі висновків.

**Sprint 4:** реалізуємо код і tuning, фіксуємо моделі, перевіряємо один пілот.

**Sprint 5:** виконуємо **повний основний експеримент**, перевіряємо дані, аналізуємо `Q`, `D`, варіативність та оцінюємо H1 у межах протоколу.

**Результат спринту:** Experimental Results v1.0 — перевірювані фактичні вимірювання, графіки, пояснення їхнього зв'язку з RQ і чесно визначені межі висновків.

**Sprint 6:** **Наступний крок** —  використати отримані результати для написання обґрунтованого дослідницького рукопису й наукової комунікації, не підмінюючи опис результатів припущеннями.